# synthetic 04 build sensor messages stage


## Overview

This notebook supports the synthetic pipeline stage documented by this notebook. It is part of the Synthetic support portion of the capstone workflow and should be read as a notebook-first implementation step rather than a separate pipeline redesign.


## Objectives

- Document how this notebook supports the synthetic pipeline stage documented by this notebook.
- Preserve the existing notebook-first execution flow, configuration usage, logger behavior, ledger behavior, and artifact outputs.
- Make the notebook easier to review by separating setup, validation, processing, outputs, and handoff context.
- Clarify whether the notebook is a support, testing, streaming, or Bronze-handoff step in the synthetic path.


## Code Reference

A detailed code-cell reference for this notebook is maintained in the project documentation:`docs/technical_reference/notebook_code_references/synthetic_04_build_sensor_messages_stage_code_reference.md`


In [ ]:
import os

from utils.database.postgres import (
    get_engine_from_env, 
    read_sql_dataframe,
)

from utils.database.chunk_stage_util import process_postgres_table_in_chunks

from utils.synthetic.pipeline.melt_stage_writer import (
    build_sensor_messages_stage,
    validate_sensor_messages_stage,
    build_sensor_messages_stage_sql_native,
)

from utils.core.env_helpers import (
    env_int, 
    env_str,
)

print("Stage 4 imports passed.")

import inspect

print("Imports passed.")
print("process_postgres_table_in_chunks:")
print(inspect.signature(process_postgres_table_in_chunks))

print("\nbuild_sensor_messages_stage:")
print(inspect.signature(build_sensor_messages_stage))

print("\nbuild_sensor_messages_stage_sql_native:")
print(inspect.signature(build_sensor_messages_stage_sql_native))


Stage 4 imports passed.
Imports passed.
process_postgres_table_in_chunks:
(engine, *, schema_name: 'str', table_name: 'str', select_columns: 'list[str]', order_by_sql: 'str', transform_chunk_func: 'Callable[[pd.DataFrame, int, int, int], Any]', write_chunk_func: 'Callable[[Any, int, int, int], None]', chunk_size: 'int' = 10000, where_sql: 'str' = '', params: 'Optional[Mapping[str, Any]]' = None, enable_memory_logging: 'bool' = False) -> 'None'

build_sensor_messages_stage:
(engine, *, schema: 'str' = 'capstone', source_table: 'str' = 'synthetic_observations_timestamped_stage', target_table: 'str' = 'synthetic_sensor_messages_stage', if_exists: 'str' = 'replace', random_seed: 'int' = 42, n_sensors: 'int' = 52, chunk_size: 'int' = 10000, enable_memory_logging: 'bool' = False) -> 'str'

build_sensor_messages_stage_sql_native:
(engine, *, schema: 'str', source_table: 'str', target_table: 'str', n_sensors: 'int' = 52, enable_memory_logging: 'bool' = False) -> 'str'


In [ ]:
SCHEMA = env_str("CAPSTONE_SCHEMA", "capstone")

DATASET_ID = env_str(
    "SYNTHETIC_DATASET_ID",
    "pump_synthetic_v1",
    aliases=("DATASET_ID",),
)

RUN_ID = env_str(
    "SYNTHETIC_RUN_ID",
    "synthetic_run_001",
    aliases=("RUN_ID",),
)

IF_EXISTS_FLAG = "replace"
RANDOM_SEED = env_int("SYNTHETIC_RANDOM_SEED", 42)
NUMBER_OF_SENSORS = env_int("SYNTHETIC_SENSOR_COUNT", 52)

CHUNK_SIZE = env_int("SYNTHETIC_MELT_SOURCE_ROW_CHUNK_SIZE", 25000)

MELT_SOURCE_TABLE_NAME = env_str(
    "SYNTHETIC_TIMESTAMPED_OBSERVATIONS_TABLE",
    "synthetic_observations_timestamped_stage",
)

MELT_DESTINATION_TABLE_NAME = env_str(
    "SYNTHETIC_SENSOR_MESSAGES_TABLE",
    "synthetic_sensor_messages_stage",
)

OBSERVATION_BATCH_SIZE = env_int("OBSERVATION_BATCH_SIZE", 500)
# One observation expands to one row per sensor after melt, so the effective batch size scales with the sensor count.
MESSAGE_BATCH_SIZE = OBSERVATION_BATCH_SIZE * NUMBER_OF_SENSORS

print("Synthetic melt config")
print("schema:", SCHEMA)
print("source table:", MELT_SOURCE_TABLE_NAME)
print("target table:", MELT_DESTINATION_TABLE_NAME)
print("source row chunk size:", CHUNK_SIZE)
print("message batch size:", MESSAGE_BATCH_SIZE)

Synthetic melt config
schema: capstone
source table: synthetic_observations_timestamped_stage
target table: synthetic_sensor_messages_stage
source row chunk size: 25000
message batch size: 26000


---

In [4]:

engine = get_engine_from_env()


---

In [5]:
# The SQL-native path avoids loading the full wide table into Python memory; the melt is done in-database as a CROSS JOIN LATERAL or UNPIVOT equivalent.
melt_table_name = build_sensor_messages_stage_sql_native(
    engine=engine,
    schema=SCHEMA,
    source_table=MELT_SOURCE_TABLE_NAME,
    target_table=MELT_DESTINATION_TABLE_NAME,
    n_sensors=NUMBER_OF_SENSORS,
    enable_memory_logging=True,
)

print("Built SQL-native melted table:", melt_table_name)

No missing sensor columns found.
[memory] stage 04 sql-native - before metadata read: 0.17 GB
[memory] stage 04 sql-native - after metadata read: 0.17 GB
[memory] stage 04 sql-native - before SQL melt/create table: 0.17 GB
[memory] stage 04 sql-native - after SQL melt/create table: 0.17 GB
[memory] stage 04 sql-native - before indexes/analyze: 0.17 GB
[memory] stage 04 sql-native - after indexes/analyze: 0.17 GB
[memory] stage 04 sql-native - after gc: 0.17 GB
Built SQL-native melted table: synthetic_sensor_messages_stage


In [6]:
# The pandas-based melt is disabled because it cannot handle the full dataset volume in the devcontainer memory limit.
'''
melt_table_name = build_sensor_messages_stage(
    engine=engine,
    schema=SCHEMA,
    source_table=MELT_SOURCE_TABLE_NAME,
    target_table=MELT_DESTINATION_TABLE_NAME,
    if_exists=IF_EXISTS_FLAG,
    random_seed=RANDOM_SEED,
    n_sensors=NUMBER_OF_SENSORS,
    chunk_size=CHUNK_SIZE,
    enable_memory_logging=True,
)
'''


'\nmelt_table_name = build_sensor_messages_stage(\n    engine=engine,\n    schema=SCHEMA,\n    source_table=MELT_SOURCE_TABLE_NAME,\n    target_table=MELT_DESTINATION_TABLE_NAME,\n    if_exists=IF_EXISTS_FLAG,\n    random_seed=RANDOM_SEED,\n    n_sensors=NUMBER_OF_SENSORS,\n    chunk_size=CHUNK_SIZE,\n    enable_memory_logging=True,\n)\n'

---

In [7]:

print("Built table:", melt_table_name)


Built table: synthetic_sensor_messages_stage


---

In [8]:

validation_dataframe = validate_sensor_messages_stage(
    engine=engine,
    schema=SCHEMA,
    table_name=MELT_DESTINATION_TABLE_NAME,
)

display(validation_dataframe)

,row_count,distinct_observation_count,distinct_observation_timestamp_count,distinct_sensor_name_count,min_sensor_index,max_sensor_index,min_message_sequence_index,max_message_sequence_index
0,11700000,225000,225000,52,0,51,0,51


---

In [9]:
# Expected row count is source observations multiplied by sensor count.
stage_04_validation_dataframe = read_sql_dataframe(
    engine,
    f"""
    WITH source_counts AS (
        SELECT
            COUNT(*) AS source_observation_count
        FROM "{SCHEMA}"."{MELT_SOURCE_TABLE_NAME}"
    ),
    message_counts AS (
        SELECT
            COUNT(*) AS message_row_count,
            COUNT(DISTINCT observation_index) AS distinct_observation_count,
            COUNT(DISTINCT sensor_name) AS distinct_sensor_name_count,
            COUNT(DISTINCT sensor_index) AS distinct_sensor_index_count,
            MIN(sensor_index) AS min_sensor_index,
            MAX(sensor_index) AS max_sensor_index,
            MIN(message_sequence_index) AS min_message_sequence_index,
            MAX(message_sequence_index) AS max_message_sequence_index
        FROM "{SCHEMA}"."{MELT_DESTINATION_TABLE_NAME}"
    )
    SELECT
        source_counts.source_observation_count,
        message_counts.message_row_count,
        message_counts.distinct_observation_count,
        message_counts.distinct_sensor_name_count,
        message_counts.distinct_sensor_index_count,
        message_counts.min_sensor_index,
        message_counts.max_sensor_index,
        message_counts.min_message_sequence_index,
        message_counts.max_message_sequence_index,
        source_counts.source_observation_count * {NUMBER_OF_SENSORS} AS expected_message_row_count,
        message_counts.message_row_count - source_counts.source_observation_count * {NUMBER_OF_SENSORS} AS message_row_delta
    FROM source_counts
    CROSS JOIN message_counts
    """
)

display(stage_04_validation_dataframe)

,source_observation_count,message_row_count,distinct_observation_count,distinct_sensor_name_count,distinct_sensor_index_count,min_sensor_index,max_sensor_index,min_message_sequence_index,max_message_sequence_index,expected_message_row_count,message_row_delta
0,225000,11700000,225000,52,52,0,51,0,51,11700000,0


----

In [10]:
# Incomplete observations would cause the consumer to receive partial sensor data for a timestamp, breaking row rebuild in Stage 9.
incomplete_observations_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        observation_index,
        COUNT(*) AS message_count,
        COUNT(DISTINCT sensor_index) AS distinct_sensor_index_count,
        COUNT(DISTINCT sensor_name) AS distinct_sensor_name_count
    FROM "{SCHEMA}"."{MELT_DESTINATION_TABLE_NAME}"
    GROUP BY observation_index
    HAVING COUNT(*) <> {NUMBER_OF_SENSORS}
        OR COUNT(DISTINCT sensor_index) <> {NUMBER_OF_SENSORS}
        OR COUNT(DISTINCT sensor_name) <> {NUMBER_OF_SENSORS}
    ORDER BY observation_index
    LIMIT 25
    """
)

display(incomplete_observations_dataframe)

,observation_index,message_count,distinct_sensor_index_count,distinct_sensor_name_count


---

In [11]:
sample_sensor_messages_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        observation_index,
        generated_row_id,
        observation_timestamp,
        sensor_name,
        sensor_index,
        sensor_value,
        message_sequence_index,
        stream_state,
        phase,
        meta_episode_id
    FROM "{SCHEMA}"."{MELT_DESTINATION_TABLE_NAME}"
    ORDER BY observation_index, sensor_index
    LIMIT 104
    """
)

display(sample_sensor_messages_dataframe)

,observation_index,generated_row_id,observation_timestamp,sensor_name,sensor_index,sensor_value,message_sequence_index,stream_state,phase,meta_episode_id
0,1,synthetic_run_001_obs_000000000001,2026-04-16 00:00:00+00:00,sensor_00,0,2.344329,0,normal,normal,0
1,1,synthetic_run_001_obs_000000000001,2026-04-16 00:00:00+00:00,sensor_01,1,48.327756,1,normal,normal,0
2,1,synthetic_run_001_obs_000000000001,2026-04-16 00:00:00+00:00,sensor_02,2,53.575950,2,normal,normal,0
3,1,synthetic_run_001_obs_000000000001,2026-04-16 00:00:00+00:00,sensor_03,3,43.424633,3,normal,normal,0
4,1,synthetic_run_001_obs_000000000001,2026-04-16 00:00:00+00:00,sensor_04,4,617.854335,4,normal,normal,0
...,...,...,...,...,...,...,...,...,...,...
99,2,synthetic_run_001_obs_000000000002,2026-04-16 00:01:00+00:00,sensor_47,47,44.544907,47,normal,normal,0
100,2,synthetic_run_001_obs_000000000002,2026-04-16 00:01:00+00:00,sensor_48,48,39.875294,48,normal,normal,0
101,2,synthetic_run_001_obs_000000000002,2026-04-16 00:01:00+00:00,sensor_49,49,54.131620,49,normal,normal,0
102,2,synthetic_run_001_obs_000000000002,2026-04-16 00:01:00+00:00,sensor_50,50,161.877550,50,normal,normal,0


----

In [12]:
sample_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        observation_index,
        observation_timestamp,
        generated_row_id,
        sensor_name,
        sensor_index,
        sensor_value,
        message_sequence_index,
        stream_state,
        phase,
        meta_episode_id
    FROM {SCHEMA}.{MELT_DESTINATION_TABLE_NAME}
    ORDER BY observation_index, message_sequence_index
    LIMIT 104
    """
)

display(sample_dataframe)

,observation_index,observation_timestamp,generated_row_id,sensor_name,sensor_index,sensor_value,message_sequence_index,stream_state,phase,meta_episode_id
0,1,2026-04-16 00:00:00+00:00,synthetic_run_001_obs_000000000001,sensor_00,0,2.344329,0,normal,normal,0
1,1,2026-04-16 00:00:00+00:00,synthetic_run_001_obs_000000000001,sensor_01,1,48.327756,1,normal,normal,0
2,1,2026-04-16 00:00:00+00:00,synthetic_run_001_obs_000000000001,sensor_02,2,53.575950,2,normal,normal,0
3,1,2026-04-16 00:00:00+00:00,synthetic_run_001_obs_000000000001,sensor_03,3,43.424633,3,normal,normal,0
4,1,2026-04-16 00:00:00+00:00,synthetic_run_001_obs_000000000001,sensor_04,4,617.854335,4,normal,normal,0
...,...,...,...,...,...,...,...,...,...,...
99,2,2026-04-16 00:01:00+00:00,synthetic_run_001_obs_000000000002,sensor_47,47,44.544907,47,normal,normal,0
100,2,2026-04-16 00:01:00+00:00,synthetic_run_001_obs_000000000002,sensor_48,48,39.875294,48,normal,normal,0
101,2,2026-04-16 00:01:00+00:00,synthetic_run_001_obs_000000000002,sensor_49,49,54.131620,49,normal,normal,0
102,2,2026-04-16 00:01:00+00:00,synthetic_run_001_obs_000000000002,sensor_50,50,161.877550,50,normal,normal,0


---

In [13]:
sequence_check_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        observation_index,
        COUNT(*) AS message_count,
        COUNT(DISTINCT sensor_index) AS distinct_sensor_count,
        COUNT(DISTINCT observation_timestamp) AS distinct_observation_timestamp_count,
        MIN(message_sequence_index) AS min_msg_seq,
        MAX(message_sequence_index) AS max_msg_seq,
        COUNT(DISTINCT message_sequence_index) AS distinct_msg_seq_count
    FROM {SCHEMA}.{MELT_DESTINATION_TABLE_NAME}
    GROUP BY observation_index
    ORDER BY observation_index
    LIMIT 10
    """
)

display(sequence_check_dataframe)

,observation_index,message_count,distinct_sensor_count,distinct_observation_timestamp_count,min_msg_seq,max_msg_seq,distinct_msg_seq_count
0,1,52,52,1,0,51,52
1,2,52,52,1,0,51,52
2,3,52,52,1,0,51,52
3,4,52,52,1,0,51,52
4,5,52,52,1,0,51,52
5,6,52,52,1,0,51,52
6,7,52,52,1,0,51,52
7,8,52,52,1,0,51,52
8,9,52,52,1,0,51,52
9,10,52,52,1,0,51,52


----

In [14]:
# Dispose Engine
engine.dispose()

## Summary

This notebook preserves the current analytical behavior while documenting the role of the Synthetic support step in the capstone workflow. The code cells above should continue to produce the same configured outputs, artifacts, logger messages, and ledger entries as before this documentation pass.


## Next Stage

Sensor messages feed the send queue used by the producer path.
